In [1]:
import pandas as pd

df = pd.read_csv("./spaceship-titanic/train.csv")
test_df = pd.read_csv("./spaceship-titanic/test.csv")

test_ids = test_df["PassengerId"]

y = df["Transported"]

train_df = df.drop("Transported", axis=1)
all_df = pd.concat([df, test_df])

print(df.head(5))
print(test_df.head(5))

  PassengerId HomePlanet CryoSleep  Cabin  Destination   Age    VIP  \
0     0001_01     Europa     False  B/0/P  TRAPPIST-1e  39.0  False   
1     0002_01      Earth     False  F/0/S  TRAPPIST-1e  24.0  False   
2     0003_01     Europa     False  A/0/S  TRAPPIST-1e  58.0   True   
3     0003_02     Europa     False  A/0/S  TRAPPIST-1e  33.0  False   
4     0004_01      Earth     False  F/1/S  TRAPPIST-1e  16.0  False   

   RoomService  FoodCourt  ShoppingMall     Spa  VRDeck               Name  \
0          0.0        0.0           0.0     0.0     0.0    Maham Ofracculy   
1        109.0        9.0          25.0   549.0    44.0       Juanna Vines   
2         43.0     3576.0           0.0  6715.0    49.0      Altark Susent   
3          0.0     1283.0         371.0  3329.0   193.0       Solam Susent   
4        303.0       70.0         151.0   565.0     2.0  Willy Santantines   

   Transported  
0        False  
1         True  
2        False  
3        False  
4         True  
  

In [2]:
df.info()
df.describe(include="all")

test_df.info()
test_df.describe(include="all")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4277 entries, 0 to 4276
Data columns (total 13 columns):
 #   Column        Non-Null Count  

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
count,4277,4190,4184,4177,4185,4186.000000,4184,4195.000000,4171.000000,4179.000000,4176.000000,4197.000000,4183
unique,4277,3,2,3265,3,NaN,2,NaN,NaN,NaN,NaN,NaN,4176
top,9277_01,Earth,False,G/160/P,TRAPPIST-1e,NaN,False,NaN,NaN,NaN,NaN,NaN,Lyney Sellahaney
freq,1,2263,2640,8,2956,NaN,4110,NaN,NaN,NaN,NaN,NaN,2
mean,NaN,NaN,NaN,NaN,NaN,28.658146,NaN,219.266269,439.484296,177.295525,303.052443,310.710031,NaN
std,NaN,NaN,NaN,NaN,NaN,14.179072,NaN,607.011289,1527.663045,560.821123,1117.186015,1246.994742,NaN
min,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
25%,NaN,NaN,NaN,NaN,NaN,19.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
50%,NaN,NaN,NaN,NaN,NaN,26.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
75%,NaN,NaN,NaN,NaN,NaN,37.000000,NaN,53.000000,78.000000,33.000000,50.000000,36.000000,NaN


In [3]:
import numpy as np

spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
all_df[spend_cols] = all_df[spend_cols].fillna(0)

all_df["TotalSpend"] = all_df[spend_cols].sum(axis=1)
all_df["NoSpend"] = all_df["TotalSpend"] == 0

# 支出カテゴリの分割
all_df["LuxurySpend"] = all_df["Spa"] + all_df["VRDeck"]
all_df["NecessitySpend"] = all_df["RoomService"] + all_df["FoodCourt"]

# 支出の対数変換（スキューを補正）
for col in spend_cols:
    all_df[f"Log_{col}"] = np.log1p(all_df[col])
all_df["LogTotalSpend"] = np.log1p(all_df["TotalSpend"])

# Cabin から Deck・番号・Side を抽出
all_df["Deck"] = all_df["Cabin"].str.split("/").str[0]
all_df["CabinNum"] = all_df["Cabin"].str.split("/").str[1].astype(float)
all_df["Side"] = all_df["Cabin"].str.split("/").str[2]

# 年齢をビンに分割
all_df["AgeBin"] = pd.cut(
    all_df["Age"].fillna(all_df["Age"].median()),
    bins=[0, 12, 18, 35, 60, 200],
    labels=["Child", "Teen", "Young", "Adult", "Senior"]
)

# グループ情報
all_df["Group"] = all_df["PassengerId"].str.split("_").str[0]
all_df["GroupSize"] = all_df.groupby("Group")["Group"].transform("count")
all_df["IsAlone"] = all_df["GroupSize"] == 1

all_df = all_df.drop(["Name", "Cabin"], axis=1)


In [4]:
transported = all_df["Transported"]
all_df_encoded = pd.get_dummies(all_df.drop("Transported", axis=1))
all_df_encoded = all_df_encoded.fillna(all_df_encoded.median(numeric_only=True)).fillna(0)

train = all_df_encoded[:len(y)].copy()
test = all_df_encoded[len(y):].copy()

train["Transported"] = transported[:len(y)].values

X = train

print(df.head(5))


  PassengerId HomePlanet CryoSleep  Cabin  Destination   Age    VIP  \
0     0001_01     Europa     False  B/0/P  TRAPPIST-1e  39.0  False   
1     0002_01      Earth     False  F/0/S  TRAPPIST-1e  24.0  False   
2     0003_01     Europa     False  A/0/S  TRAPPIST-1e  58.0   True   
3     0003_02     Europa     False  A/0/S  TRAPPIST-1e  33.0  False   
4     0004_01      Earth     False  F/1/S  TRAPPIST-1e  16.0  False   

   RoomService  FoodCourt  ShoppingMall     Spa  VRDeck               Name  \
0          0.0        0.0           0.0     0.0     0.0    Maham Ofracculy   
1        109.0        9.0          25.0   549.0    44.0       Juanna Vines   
2         43.0     3576.0           0.0  6715.0    49.0      Altark Susent   
3          0.0     1283.0         371.0  3329.0   193.0       Solam Susent   
4        303.0       70.0         151.0   565.0     2.0  Willy Santantines   

   Transported  
0        False  
1         True  
2        False  
3        False  
4         True  


In [5]:
X = train.drop("Transported", axis=1)
y = train["Transported"].astype(bool)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
import lightgbm as lgb

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# XGBoost
xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss",
    early_stopping_rounds=30
)
xgb_model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
print("XGBoost Accuracy:", accuracy_score(y_valid, xgb_model.predict(X_valid)))

# LightGBM
lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
lgb_model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)],
              callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
print("LightGBM Accuracy:", accuracy_score(y_valid, lgb_model.predict(X_valid)))

# 良い方を model に設定
model = xgb_model if accuracy_score(y_valid, xgb_model.predict(X_valid)) >= accuracy_score(y_valid, lgb_model.predict(X_valid)) else lgb_model
print("Using:", type(model).__name__)


XGBoost Accuracy: 0.7981598619896493
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 3500, number of negative: 3454
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000842 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3964
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 43
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503307 -> initscore=0.013230
[LightGBM] [Info] Start training from score 0.013230
Training until validation scores don't improve for 30 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits 

In [6]:
pred = model.predict(X_valid)

score = accuracy_score(y_valid, pred)

print("Accuracy:", score)

pred = model.predict(test)

submission = pd.DataFrame({
    "PassengerId":test_ids,
    "Transported":pred
})

submission.to_csv("subnission.csv", index=False)


Accuracy: 0.8033352501437608


In [7]:
from xgboost import XGBClassifier
import lightgbm as lgb

if isinstance(model, XGBClassifier):
    importance = pd.Series(model.feature_importances_, index=X_train.columns)
else:
    importance = pd.Series(model.feature_importances_, index=X_train.columns)
print(importance.sort_values(ascending=False).head(15))


CabinNum          705
LuxurySpend       495
Age               434
FoodCourt         382
TotalSpend        353
Spa               250
ShoppingMall      249
RoomService       240
Side_P            195
VRDeck            147
NecessitySpend    136
GroupSize         109
Deck_E            104
Deck_C             98
LogTotalSpend      92
dtype: int32
